In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import logging
logger = logging.getLogger()
logger.setLevel(logging.CRITICAL)

In [3]:
import json
from IPython.display import JSON
import os
from dotenv import load_dotenv

from unstructured_client import UnstructuredClient
from unstructured_client.models import shared
from unstructured_client.models.errors import SDKError
from unstructured_client.models import operations
from unstructured.chunking.basic import chunk_elements
from unstructured.chunking.title import chunk_by_title
from unstructured.staging.base import dict_to_elements

import chromadb

In [4]:
load_dotenv()

api_key = os.getenv("UNSTRUCTURED_API_KEY")

In [5]:
from Utils import Utils
from unstructured_client import UnstructuredClient

utils = Utils()

API_KEY = utils.get_api_key()
API_URL = utils.get_url()

s = UnstructuredClient(
    api_key_auth=API_KEY,
    server_url=API_URL,
)

In [6]:
filename = "transformers.pdf"
with open(filename, "rb") as f:
    files=shared.Files(
        content=f.read(),
        file_name=filename,
        strategy="ocr_only",
        infer_table_structure=False
    )

req = shared.PartitionParameters(files=files)

In [7]:
try:
    response = s.general.partition(
        request=operations.PartitionRequest(
            partition_parameters=req
        )
    )
except SDKError as e:
    print(e)

INFO: HTTP Request: GET https://api.unstructuredapp.io/general/docs "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"
INFO: HTTP Request: POST https://api.unstructuredapp.io/general/v0/general "HTTP/1.1 200 OK"


In [8]:
JSON(json.dumps(response.elements[0:3], indent=2))

<IPython.core.display.JSON object>

In [9]:
[x for x in response.elements if x['type'] == 'Title']

[{'type': 'Title',
  'element_id': '80340048f8a9fcdd6943f2ae57e4579a',
  'text': 'Attention Is All You Need',
  'metadata': {'is_extracted': 'true',
   'filetype': 'application/pdf',
   'languages': ['eng'],
   'page_number': 1,
   'filename': 'transformers.pdf'}},
 {'type': 'Title',
  'element_id': '5135fcce554c88a12f632b484daeebc5',
  'text': 'Abstract',
  'metadata': {'is_extracted': 'true',
   'filetype': 'application/pdf',
   'languages': ['eng'],
   'page_number': 1,
   'filename': 'transformers.pdf',
   'parent_id': '8baebb8d0d0908107755e1c77b724cd0'}},
 {'type': 'Title',
  'element_id': 'e351bbc6bf30079b2ade8d430673db25',
  'text': '1 Introduction',
  'metadata': {'is_extracted': 'true',
   'filetype': 'application/pdf',
   'languages': ['eng'],
   'page_number': 2,
   'filename': 'transformers.pdf'}},
 {'type': 'Title',
  'element_id': '950ecf3f58d3e1f788daab7a7349538e',
  'text': '2 Background',
  'metadata': {'is_extracted': 'true',
   'filetype': 'application/pdf',
   'lang

In [10]:
[x for x in response.elements
 if x['type'] == 'Title' and 'architecture' in x['text'].lower()]

[{'type': 'Title',
  'element_id': 'd4bc3f81f1d9087a603290bf88f2c52f',
  'text': '3 Model Architecture',
  'metadata': {'is_extracted': 'true',
   'filetype': 'application/pdf',
   'languages': ['eng'],
   'page_number': 2,
   'filename': 'transformers.pdf'}}]

In [11]:
chapters = [
    "Attention Is All You Need",
    "1 Introduction",
    "2 Background",
    "3 Model Architecture",
    "4 Why Self-Attention",
    "5 Training",
    "6 Results",
    "7 Conclusion"
]

In [12]:
chapter_ids = {}
for element in response.elements:
    for chapter in chapters:
        if (
            element["type"] == "Title"
            and chapter.lower() in element["text"].lower()
        ):
            chapter_ids[element["element_id"]] = chapter
            break

In [13]:
chapter_ids

{'80340048f8a9fcdd6943f2ae57e4579a': 'Attention Is All You Need',
 'e351bbc6bf30079b2ade8d430673db25': '1 Introduction',
 '950ecf3f58d3e1f788daab7a7349538e': '2 Background',
 'd4bc3f81f1d9087a603290bf88f2c52f': '3 Model Architecture',
 '176c01580b2a2a268955111360b7e791': '4 Why Self-Attention',
 'f223ae03c4f8c9fc5b5bc5a455f0dcd3': '5 Training',
 '4195e2b1667c533894824e088d69224a': '6 Results',
 'cdc53e0213d0ceff43cf2fd66332491f': '7 Conclusion'}

In [14]:
chapter_to_id = {v: k for k, v in chapter_ids.items()}

print(chapter_to_id)

{'Attention Is All You Need': '80340048f8a9fcdd6943f2ae57e4579a', '1 Introduction': 'e351bbc6bf30079b2ade8d430673db25', '2 Background': '950ecf3f58d3e1f788daab7a7349538e', '3 Model Architecture': 'd4bc3f81f1d9087a603290bf88f2c52f', '4 Why Self-Attention': '176c01580b2a2a268955111360b7e791', '5 Training': 'f223ae03c4f8c9fc5b5bc5a455f0dcd3', '6 Results': '4195e2b1667c533894824e088d69224a', '7 Conclusion': 'cdc53e0213d0ceff43cf2fd66332491f'}


In [15]:
[x for x in response.elements
 if x["metadata"].get("parent_id") == chapter_to_id["3 Model Architecture"]]

[{'type': 'NarrativeText',
  'element_id': 'b5d6b35818646075642f4918425abab6',
  'text': 'Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) to a sequence of continuous representations z = (z1,...,zn). Given z, the decoder then generates an output sequence (y1,...,ym) of symbols one element at a time. At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next.',
  'metadata': {'is_extracted': 'true',
   'filetype': 'application/pdf',
   'languages': ['eng'],
   'page_number': 2,
   'filename': 'transformers.pdf',
   'parent_id': 'd4bc3f81f1d9087a603290bf88f2c52f'}}]

Load Data into VectorDB

In [16]:
client = chromadb.PersistentClient(path="chroma_tmp", settings=chromadb.Settings(allow_reset=True))
client.reset()

True

In [17]:
collection = client.create_collection(
    name="Transformers",
    metadata={"hnsw:space": "cosine"}
)

In [18]:
for element in response.elements:
    parent_id = element["metadata"].get("parent_id")
    chapter = chapter_ids.get(parent_id, "")
    collection.add(
        documents=[element["text"]],
        ids=[element["element_id"]],
        metadatas=[{"chapter": chapter}]
    )

## See the elements in Vector DB

In [19]:
results = collection.peek()
print(results["documents"])

['Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.', 'Attention Is All You Need', '3 2 0 2 g u A 2 ] L C . s c [ 7 v 2 6 7 3 0 . 6 0 7 1 : v i X r a', 'Ashish Vaswani∗ Google Brain avaswani@google.com', 'Llion Jones∗ Google Research llion@google.com', 'Noam Shazeer∗ Google Brain noam@google.com', 'Niki Parmar∗ Google Research nikip@google.com', 'Jakob Uszkoreit∗ Google Research usz@google.com', 'Aidan N. Gomez∗ † University of Toronto aidan@cs.toronto.edu', 'Łukasz Kaiser∗ Google Brain lukaszkaiser@google.com']


### hybrid search with metadata

In [20]:
result = collection.query(
    query_texts=["What is self attention?"],
    n_results=2,
    where={"chapter": "3 Model Architecture"}
)

print(json.dumps(result, indent=2))

{
  "ids": [
    [
      "b5d6b35818646075642f4918425abab6"
    ]
  ],
  "embeddings": null,
  "documents": [
    [
      "Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 35]. Here, the encoder maps an input sequence of symbol representations (x1,...,xn) to a sequence of continuous representations z = (z1,...,zn). Given z, the decoder then generates an output sequence (y1,...,ym) of symbols one element at a time. At each step the model is auto-regressive [10], consuming the previously generated symbols as additional input when generating the next."
    ]
  ],
  "uris": null,
  "included": [
    "metadatas",
    "documents",
    "distances"
  ],
  "data": null,
  "metadatas": [
    [
      {
        "chapter": "3 Model Architecture"
      }
    ]
  ],
  "distances": [
    [
      0.895134449005127
    ]
  ]
}


###Chunking

In [21]:
elements = dict_to_elements(response.elements)

In [22]:
chunks = chunk_by_title(
    elements,
    combine_text_under_n_chars=100,
    max_characters=3000,
)

In [23]:
JSON(json.dumps(chunks[0].to_dict(), indent=2))

<IPython.core.display.JSON object>

In [24]:
len(elements)

191

In [25]:
len(chunks)

28